# Python For ML Session 7 : Workbook on ML workflow

In **Session 5** you saw the machine-learning workflow run end to end (load → split → fit → predict → score), and in **Session 6** you saw how a raw, messy table gets *cleaned* before a model ever touches it.

**You will complete the workflow discussed in those session yourself in this session.** You'll take a brand-new, messy dataset from a raw CSV full of gaps and text columns to a working, evaluated model. The steps are exactly the ones you've already seen; the dataset is new so you have to reinterpret the concepts we discussed previously to this new dataset.

Every task below gives you a short reminder and the general syntax pattern. You've seen all of these before, you just have to apply it to a new context.

## The dataset: `auto-mpg`

We'll use the **auto-mpg** dataset: technical specs for ~400 cars from the 1970s–80s. It comes from the same place as the penguins data, so loading it is familiar.

Each row is one car model, with columns like:

| Column | Meaning |
|---|---|
| `mpg` | fuel efficiency (miles per gallon) — higher is better |
| `cylinders`, `displacement`, `horsepower`, `weight`, `acceleration` | engine / body measurements |
| `model_year` | year of the model (70 = 1970) |
| `origin` | where it was built: `usa`, `japan`, or `europe` |
| `name` | the car's full name (free text) |

This dataset is also a messy dataset similar to titanic dataset, which will give you a practice similar to something you would have in the real world. It has a numeric column with **missing values**, a **text** column that a model can't read, a useless free-text column, and numbers on **wildly different scales** (`weight` is in the thousands, `acceleration` is in the tens). Perfect for practising the whole pipeline.

We'll predict two different things with it:
- **Classification:** predict a car's `origin` from its specs.
- **Regression:** predict a car's `mpg` from its specs.

## The workflow at a glance

You're combining the two checklists you already know:

**Clean the data (Session 6):**
1. **Inspect** : what's there, what's missing, what's text vs numbers
2. **Handle missing values** : drop or fill
3. **Encode** text columns into numbers : *only the columns you'll use as inputs*
4. **Handle outliers** : extreme values
5. **Scale** features : *only for distance/gradient models, not trees*

**Build the model (Session 5):**

6. **Choose** features `X` and label `y`
7. **Split** into train / test
8. **Train** (`fit`), **Predict** (`predict`), **Evaluate** (`score`)

> One thing to notice as you go: steps 3 (encode) and 5 (scale) depend on **what you're predicting** and **which model** you use. That's why we apply them at the moment they're actually needed, not all up front.

## Part A : Clean the dataset

These cleaning steps don't depend on what we predict, so we do them once, first.

### Step 0 : Setup and load

Just like Session 5, we import our tools and load the CSV straight from a URL.

In [1]:
# Run this cell as-is to get your tools ready.
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, r2_score, mean_squared_error

DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv"

### Task 1: Load and inspect

You can't clean what you haven't looked at. Load the data, then use the three inspection tools from Session 6.

Reminder of the pattern:

```python
df = pd.read_csv(DATA_URL)
df.head()
df.shape
df.info()              # column types + how many non-null values each has
df.isnull().sum()      # exactly how many missing values per column
df.describe()          # ranges and means — watch min vs max for outlier hints
```

In [3]:
# Task 1
# Load the CSV from DATA_URL into a DataFrame called df
df = pd.read_csv(DATA_URL)


# Print the shape (rows, columns)
print(df.shape)


# Show the first few rows
print(df.head())

# Show how many missing values are in each column

print(df.isnull().sum())


# Show describe() to see the ranges (look at the min and max of each column)
print(df.describe())

(398, 9)
    mpg  cylinders  displacement  horsepower  weight  acceleration  \
0  18.0          8         307.0       130.0    3504          12.0   
1  15.0          8         350.0       165.0    3693          11.5   
2  18.0          8         318.0       150.0    3436          11.0   
3  16.0          8         304.0       150.0    3433          12.0   
4  17.0          8         302.0       140.0    3449          10.5   

   model_year origin                       name  
0          70    usa  chevrolet chevelle malibu  
1          70    usa          buick skylark 320  
2          70    usa         plymouth satellite  
3          70    usa              amc rebel sst  
4          70    usa                ford torino  
mpg             0
cylinders       0
displacement    0
horsepower      6
weight          0
acceleration    0
model_year      0
origin          0
name            0
dtype: int64
              mpg   cylinders  displacement  horsepower       weight  \
count  398.000000  398.

**Before you move on, answer for yourself (a comment is fine):**
- Which column has missing values, and how many?
- Which columns are *text* (object) rather than numbers?
- Which column looks like it has a very large maximum compared to its average?

### Task 2: Handle the missing values

A model cannot train on `NaN`. From Session 6: for a **numeric** column, filling with the **median** is the safe default (it's robust to skew, unlike the mean).

First, look at how far apart the mean and median are, a big gap is a clue the column is skewed, which is exactly when the median is the better choice.

```python
print(df["some_col"].mean(), df["some_col"].median())
df["some_col"] = df["some_col"].fillna(df["some_col"].median())
```

In [4]:
# Task 2
# Print the mean and the median of the column that has missing values.
# How far apart are they?

mean = df["horsepower"].mean()
median = df["horsepower"].median()

print(mean, median)
print(mean - median)


# Fill that column's missing values with its median

df["horsepower"] = df["horsepower"].fillna(median)

# Confirm there are no missing values left (isnull().sum() should be all zeros)
print(df.isnull().sum())

104.46938775510205 93.5
10.969387755102048
mpg             0
cylinders       0
displacement    0
horsepower      0
weight          0
acceleration    0
model_year      0
origin          0
name            0
dtype: int64


### Task 3: Drop the column that can't help

One column is free text that's different for almost every row (the car's full name). A column with a unique value per row carries no general pattern a model can learn, so we drop it.

```python
df = df.drop(columns=["column_name"])
```

In [5]:
# Task 3
df.head()
# Drop the free-text name column from df

df = df.drop(columns=["name"])


# Print the remaining column names to confirm it's gone
df.head()


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,18.0,8,307.0,130.0,3504,12.0,70,usa
1,15.0,8,350.0,165.0,3693,11.5,70,usa
2,18.0,8,318.0,150.0,3436,11.0,70,usa
3,16.0,8,304.0,150.0,3433,12.0,70,usa
4,17.0,8,302.0,140.0,3449,10.5,70,usa


### Task 4: Detect and handle outliers

From Session 6, the **IQR rule** flags values far outside the normal range. We'll check `horsepower` (a few muscle cars have huge engines).

```python
q1 = df["col"].quantile(0.25)
q3 = df["col"].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr
```

Then **cap** (clip) the extremes to the bounds rather than deleting the rows — gentler, and it keeps the data:

```python
df["col"] = df["col"].clip(lower=lower, upper=upper)
```

Remember the caution from Session 6: an outlier isn't automatically an error. A genuinely powerful engine is real. The problem is that including outlier values hinders our model's generalizability.

In [6]:
# Task 4
# Compute Q1, Q3, IQR, and the upper/lower bounds for the "horsepower" column

q1 = df["displacement"].quantile(0.25)
q3 = df["displacement"].quantile(0.75)
iqr = q3-q1

print(q1)
print(q3)
print(iqr)

# Count how many horsepower values fall outside [lower, upper] and print it
lower = q1 - 1.5*iqr
upper = q3 + 1.5*iqr
outliers = df[(df["displacement"] < lower) | (df["displacement"] > upper)].shape[0]
print(outliers)

# Cap the horsepower column to the bounds using .clip(...)
df["displacement"] = df["displacement"].clip(lower, upper)

# Print horsepower's describe() before and after if you want to see the effect
print("\nAfter capping:")
print(df["displacement"].describe())

104.25
262.0
157.75
0

After capping:
count    398.000000
mean     193.425879
std      104.269838
min       68.000000
25%      104.250000
50%      148.500000
75%      262.000000
max      455.000000
Name: displacement, dtype: float64


At this point `df` is **gap-free**, the junk column is gone, and the extreme engines are tamed. The only text column left is `origin`. What we do with `origin` next depends on whether it's the thing we're *predicting* or an *input*, so we handle it inside each modelling part below.

## Part B: **Classification** - predict `origin`

For this part pick `origin` (`usa` / `japan` / `europe`) as the **label**, the thing we predict. scikit-learn is happy to predict a text label directly, so we **don't** encode it. Our features are all the numeric columns.

### Task 5: Build X and y, scale, split, train, evaluate

This is the whole Session 5 workflow in one task. Take it one line at a time.

**Features and label:**
```python
X = df.drop(columns=["species"])   # everything except the label
y = df["species"]                  # the label (left as text)
```

**Scale the features.** `origin` will be predicted by models that may care about scale, so we put the numeric columns on a comparable footing with `StandardScaler` (mean 0, similar spread):
```python
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
```

**Split, train a decision tree, score** (exactly as in Session 5):
```python
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
model = DecisionTreeClassifier(max_depth=4, random_state=0)
model.fit(X_train, y_train)
model.score(X_test, y_test)
```

In [7]:
# Task 5
# Build X (all columns except "origin") and y (the "origin" column)
x = df.drop(columns=["origin"])
y = df["origin"]

# Scale the features with StandardScaler into X_scaled
scaler = StandardScaler()
x_scaled = pd.DataFrame(scaler.fit_transform(x), columns=x.columns)

# Split into train/test (test_size=0.2, random_state=42)
x_train, x_test, y_train, y_test = train_test_split(x_scaled, y, test_size=0.2, random_state=42)


# Create a DecisionTreeClassifier(max_depth=4, random_state=0) and fit it on the training data
model = DecisionTreeClassifier(max_depth=4, random_state=0)
model.fit(x_train, y_train)

# Print the accuracy on the test set with model.score(...)
print("Accuracy on test set:", model.score(x_test, y_test))

Accuracy on test set: 0.775


You should land somewhere around **0.75–0.80** accuracy. What we are doing here is predicting the source of the car purely from its specs.

If you want to see *which* origins get confused with which, print a confusion matrix (Session 5):

```python
preds = model.predict(X_test)
print(confusion_matrix(y_test, preds, labels=model.classes_))
print(list(model.classes_))
```

### Task 6: Swap the model

The best part of this workflow: to try a different model, **only one line changes.** Swap the decision tree for a `KNeighborsClassifier` and compare. You have already preprocessed the data, just use it for the new model.

```python
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
knn.score(X_test, y_test)
```

In [9]:
# Task 6
# Train a KNeighborsClassifier(n_neighbors=5) on the SAME X_train, y_train
x = df.drop(columns=["origin"])
y = df["origin"]
scaler = StandardScaler()
x_scaled = pd.DataFrame(scaler.fit_transform(x), columns=x.columns)
x_train, x_test, y_train, y_test = train_test_split(x_scaled, y, test_size=0.2, random_state=42)
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(x_train, y_train)

# Print its test accuracy and compare it to the decision tree
print("KNN Accuracy on test set:", knn_model.score(x_test, y_test))

KNN Accuracy on test set: 0.7125


### Task 7 (investigate): Does scaling actually matter here?

Scaling can make a huge difference for distance-based models like KNN. Does it here? Find out for yourself.

- Rebuild `X_train`/`X_test` from the **unscaled** `X` (not `X_scaled`), using the same split settings.
- Train the **decision tree** on scaled vs unscaled features. Does its accuracy change?
- Train **KNN** on scaled vs unscaled features. Does *its* accuracy change?

```python
Xu_train, Xu_test, yu_train, yu_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

In [ ]:
# Task 7
# Split the UNSCALED X into train/test with the same settings
x_train_unscaled, x_test_unscaled, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# Compare the decision tree's accuracy: scaled vs unscaled features
dt_model_unscaled = DecisionTreeClassifier(max_depth=4, random_state=0)
dt_model_unscaled.fit(x_train_unscaled, y_train)
print("Decision Tree Accuracy (Unscaled):", dt_model_unscaled.score(x_test_unscaled, y_test))

# Compare KNN's accuracy: scaled vs unscaled features
knn_model_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_model_unscaled.fit(x_train_unscaled, y_train)
print("KNN Accuracy (Unscaled):", knn_model_unscaled.score(x_test_unscaled, y_test))

# What did you find? (Write your conclusion as a comment.)
 # The decision tree's accuracy is the same whether we use scaled or unscaled features, 
 # while KNN's accuracy is much higher with scaled features. This is because KNN relies on distance calculations, 
 # which can be skewed by unscaled features, while decision trees are not affected by feature scaling.

Decision Tree Accuracy (Unscaled): 0.775
KNN Accuracy (Unscaled): 0.7125


**The lesson (think before reading on):** the decision tree's score shouldn't change at all when you scale. Trees split on thresholds, so the *size* of a number is irrelevant.

Scaling is **data-dependent**, some dataset is better suited for scaling whereas some datasets are not so suited. The penguins dataset needed it badly, due to difference in scales of multiple important features; this dataset might not. The habit (scale for distance/gradient models, check whether it helped) is what matters, not a guaranteed jump in the score.

## Part C: **Regression** - predicting `mpg`

Now we predict a **number** (fuel efficiency) instead of a category — that's regression, with `LinearRegression` (the `y = wx + b` idea from the slides).

Here's the twist that ties Part A and Part B together: this time `origin` is **not** the label — it's an **input feature**. And a model can't do arithmetic on the word `"japan"`. So *now* we must **encode** it. The same column we left alone as a label in Part B has to be one-hot encoded when it becomes a feature.

### Task 8 : Encode `origin`, then run the regression

**One-hot encode** the `origin` column (Session 6):
```python
df = pd.get_dummies(df, columns=["species"], drop_first=True, dtype=int)
```
This replaces the text `spcies` with 0/1 columns like `species_adelie`, `species_gentoo`. (`drop_first=True` drops one to avoid redundancy.)

Since you are now using mpg as the label, you must create a new X and y. **Then the usual workflow**, with `body_mass_g` as the numeric label:
```python
X = df_reg.drop(columns=["body_mass_g"])
y = df_reg["body_mass_g"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
reg = LinearRegression()
reg.fit(X_train, y_train)
preds = reg.predict(X_test)
```

**Grade it with regression metrics** (Session 5): R² (1.0 is perfect) and RMSE (typical error, here in mpg):
```python
print("R^2 :", round(r2_score(y_test, preds), 3))
print("RMSE:", round(mean_squared_error(y_test, preds) ** 0.5, 2), "gram")
```

In [11]:
# Task 8
# One-hot encode the "origin" column into a new DataFrame df_reg
df_reg = pd.get_dummies(df, columns=["origin"], drop_first=True)

# Build X (everything except "mpg") and y ("mpg")
x = df_reg.drop(columns=["mpg"])
y = df_reg["mpg"]

# Split, create a LinearRegression, fit it, and predict on the test set

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
lr_model = LinearRegression()
lr_model.fit(x_train, y_train)
y_pred = lr_model.predict(x_test)

# Print R^2 and RMSE
print("R^2 Score:", lr_model.score(x_test, y_test))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

R^2 Score: 0.8449096332762083
RMSE: 2.887673367716928


Note that we did **not** scale the features for the linear regression and it worked fine, `LinearRegression` finds a coefficient per feature, so it can handle different scales on its own. Scaling matters more for KNN and for the gradient-descent training which we will talk about later.

## Recap:

Starting from a raw CSV with gaps, text, and wild scales, you:

| Step | What you did | Tool |
|---|---|---|
| Inspect | looked before cleaning | `info`, `isnull().sum()`, `describe` |
| Missing values | filled `horsepower` | `fillna(median)` |
| Junk column | dropped `name` | `drop(columns=...)` |
| Outliers | capped `horsepower` | IQR rule + `clip` |
| Encode | turned `origin` into numbers *(only when it was a feature)* | `pd.get_dummies(..., drop_first=True)` |
| Scale | put features on equal footing *(for the distance model)* | `StandardScaler` |
| Classification | predicted `origin` | `DecisionTreeClassifier`, `KNeighborsClassifier` |
| Regression | predicted `mpg` | `LinearRegression` |
| Evaluate | graded on unseen data | `score`, `accuracy_score`, `r2_score`, RMSE |

**Two big ideas to take away:**
1. The modelling workflow (split → fit → predict → score) is the **same** no matter the model or the task — you just swap one line.
2. Preprocessing is **not** a fixed button. *Which* steps you apply depends on the task (is `origin` a label or a feature?) and the model (does it care about scale?). Inspect first, decide, then clean.

# Well Done!